# MIT News: full-archive article bodies, 2021-2025

Reads `./Data/MIT_Full_Archive_Links_2021_2025.csv` and scrapes each article with the same selectors as `MIT_02_Scrape_Articles.ipynb` (all MIT News articles share one page template). Adds frequent checkpoints and resume support because the full archive is much larger than the AI-tagged corpus.

Output: `./Data/MIT_Full_Archive_Articles_2021_2025.csv`


In [ ]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

DATA_DIR = Path("./Data")
LINKS_PATH = DATA_DIR / "MIT_Full_Archive_Links_2021_2025.csv"
OUTPUT_PATH = DATA_DIR / "MIT_Full_Archive_Articles_2021_2025.csv"
CHECKPOINT_EVERY = 100

DATA_DIR.mkdir(parents=True, exist_ok=True)

links = pd.read_csv(LINKS_PATH)
print(f"Links to scrape: {len(links)}")

In [ ]:
columns = ["Published Date", "Author", "Source", "Article Header",
           "Sub_Headings", "Article Body", "Url"]
records = []
already_done_urls = set()
if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    records = existing[columns].to_dict("records")
    already_done_urls = set(existing["Url"])
    print(f"Resumed: {len(already_done_urls)} articles already scraped.")

def scrape_article(soup, url):
    record = dict.fromkeys(columns, "")
    record["Url"] = url

    header = soup.find("div", {"id": "block-mit-page-title"})
    header = header.find("span") if header else None
    if header:
        record["Article Header"] = header.get_text()

    author = soup.find("span", {"class": "news-article--author"})
    if author:
        record["Author"] = author.get_text()

    source = soup.find("span", {"class": "news-article--source"})
    if source:
        record["Source"] = source.get_text()

    date_container = soup.find("div", {"class": "news-article--publication-date"})
    date_tag = date_container.find("time") if date_container else None
    if date_tag:
        record["Published Date"] = date_tag.get_text()

    body_tag = soup.find(
        "div",
        {"class": "paragraph paragraph--type--content-block-text paragraph--view-mode--default"},
    )
    if body_tag:
        record["Article Body"] = [p.get_text().strip() for p in body_tag.find_all("p")]

    sub_heading = soup.find("div", {"class": "news-article--dek"})
    if sub_heading:
        span = sub_heading.find("span")
        record["Sub_Headings"] = span.get_text() if span else sub_heading.get_text()

    return record

def save_checkpoint():
    frame = pd.DataFrame(records, columns=columns)
    frame.to_csv(OUTPUT_PATH, index=False)
    return frame

In [ ]:
for index, url in enumerate(tqdm(links["Link"])):
    if url in already_done_urls:
        continue
    try:
        response = requests.get(url, timeout=15)
    except requests.exceptions.RequestException as exc:
        print(f"[{index}] request failed: {url} -> {exc}")
        continue
    if response.status_code != 200:
        print(f"[{index}] HTTP {response.status_code}: {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    body_tag = soup.find("body")
    if body_tag and body_tag.get("class") and body_tag["class"][0] == "page-not-found":
        continue

    records.append(scrape_article(soup, url))
    if (index + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint()

articles = save_checkpoint()
print(f"Done: {len(articles)} articles saved to {OUTPUT_PATH}")
articles.info()